### Conversation Q&A Chatbot
In many Q&A applications we want to allow the user to have a back-and-forth conversation, meaning the application needs some sort of "memory" of past questions and answers, and some logic for incorporating those into its current thinking.

In this guide we focus on adding logic for incorporating historical messages. Further details on chat history management is covered in the previous videos.

We will cover two approaches:

- Chains, in which we always execute a retrieval step;
- Agents, in which we give an LLM discretion over whether and how to execute a retrieval step (or multiple steps).

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq

groq_api_key=os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="Llama3-8b-8192")

GroqError: The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable

In [ ]:
os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [15]:
import warnings
warnings.filterwarnings("ignore")
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.llms import Ollama
embeddings=(OllamaEmbeddings(model="gemma:2b"))  ##by default it ues llama2. gemma:2b is downloaded in local pc
llm = Ollama(model="gemma:2b")
# Alternatively getting embeddings from Ollama


In [6]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

In [7]:
# 1. Load, chunk and index the contents of the blog to create a retriever.
import bs4
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

docs=loader.load()
docs

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='\n\n      LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview#\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistake

In [12]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splits=text_splitter.split_documents(docs)
vectorstore=Chroma.from_documents(documents=splits,embedding=embeddings)
retriever=vectorstore.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x12c117b60>, search_kwargs={})

In [13]:
## Prompt Template - Candidate for promopt engineering
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [16]:
question_answer_chain = create_stuff_documents_chain(llm,prompt)
# create_retrieval_chain will combine everything.
rag_chain = create_retrieval_chain(retriever,question_answer_chain)

In [17]:
response=rag_chain.invoke({"input":"What is Self-Reflection"})
response

{'input': 'What is Self-Reflection',
 'context': [Document(id='bb2b5a63-bea9-458f-bcf9-803c6902a0da', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Comparison of AD, ED, source policy and RL^2 on environments that require memory and exploration. Only binary reward is assigned. The source policies are trained with A3C for "dark" environments and DQN for watermaze.(Image source: Laskin et al. 2023)\n\nComponent Two: Memory#\n(Big thank you to ChatGPT for helping me draft this section. I’ve learned a lot about the human brain and data structure for fast MIPS in my conversations with ChatGPT.)\nTypes of Memory#\nMemory can be defined as the processes used to acquire, store, retain, and later retrieve information. There are several types of memory in human brains.\n\n\nSensory Memory: This is the earliest stage of memory, providing the ability to retain impressions of sensory information (visual, auditory, etc) after the original stimuli have end

In [19]:
response['answer'].split('\n')

["Sure, here's the answer to your question:",
 '',
 'The passage explains that memory is a process used by an agent to acquire, store, retain, and later retrieve information. The passage also mentions that the memory stream is a long-term memory module (external database) that records a comprehensive list of agents’ experience in natural language.']

In [20]:
rag_chain.invoke({"input":"Howw do we achieve it"})['answer']

'The context does not provide any information about how to achieve the goal, so I cannot answer this question from the provided context.'

### Adding Chat History

In [28]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [29]:
history_aware_retriever=create_history_aware_retriever(llm,retriever,contextualize_q_prompt)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x12c117b60>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], t

In [30]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

In [31]:
question_answer_chain=create_stuff_documents_chain(llm,qa_prompt)
rag_chain=create_retrieval_chain(history_aware_retriever,question_answer_chain)

In [32]:
from langchain_core.messages import AIMessage,HumanMessage
chat_history=[]
question="What is Self-Reflection"
response1=rag_chain.invoke({"input":question,"chat_history":chat_history})

chat_history.extend(
    [
        HumanMessage(content=question),
        AIMessage(content=response1["answer"])
    ]
)

question2="Tell me more about it?"
response2=rag_chain.invoke({"input":question,"chat_history":chat_history})
print(response2['answer'])

Sure, here's the answer to your question:

Memory is a component of the agent system that allows it to store and access past experiences and facilitate retrieval of relevant information for future behavior. Self-reflection is a framework that equips agents with dynamic memory and self-reflection capabilities to improve reasoning skills.


In [20]:
chat_history

[HumanMessage(content='What is Self-Reflection'),
 AIMessage(content='Self-Reflection is a mechanism that allows autonomous agents to improve iteratively by refining past action decisions and correcting previous mistakes. It plays a crucial role in real-world tasks where trial and error are inevitable.')]

In [34]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

In [22]:
conversational_rag_chain.invoke(
    {"input": "What is Task Decomposition?"},
    config={
        "configurable": {"session_id": "abc123"}
    },  # constructs a key "abc123" in `store`.
)["answer"]

'Task Decomposition is a technique for breaking down a complex task into smaller, more manageable steps. It helps the model to understand the task better and plan its execution more effectively.'

In [24]:
conversational_rag_chain.invoke(
    {"input": "What are common ways of doing it?"},
    config={"configurable": {"session_id": "abc123"}},
)["answer"]

'The context does not provide any information about common ways of doing it, so I cannot answer this question from the provided context.'

### Creating conversation history with llm and trying to store and study

In [71]:
from langchain_community.document_loaders.notebook import remove_newlines
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('../playground/ip_data/attention-is-all-you-need.pdf')
documents = loader.load()

splits = text_splitter.split_documents(documents)

cleaned_docs = [
    Document(page_content=remove_newlines(doc.page_content), metadata=doc.metadata)
    for doc in splits
]

In [73]:
prompt = ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>


Chat History:
{chat_history}

Question: {input}

"""
)

document_chain=create_stuff_documents_chain(llm,prompt)
# ques = 'How many people are there in the story ?'
# ques = 'Should Clara follow the young man or the older man ?'
ques = 'Summarize the content'

response = document_chain.invoke({
"input" : ques,
"context" :cleaned_docs,
"chat_history" : None,
})
# response
response.split('\n')


['The passage provides an in-depth overview of a deep learning model called "Transformer," which is widely used in natural language processing (NLP) for machine translation. ',
 '',
 'The Transformer model utilizes a novel self-attention mechanism to learn the contextual relationships between words in a sentence. This allows it to produce accurate translations of languages with high fidelity and fluency.',
 '',
 'The model consists of an encoder and a decoder, each containing multiple layers of self-attention. These layers enable the model to capture long-range dependencies and contextually integrate information from various parts of the input sequence.',
 '',
 'The passage also highlights the effectiveness of the Transformer model on various NLP tasks, including machine translation, text summarization, and question answering.',
 '',
 '']

In [74]:
# Trying to create history for the above pipeline  :
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

session_id = 'abc123'

document_chain_with_hist = RunnableWithMessageHistory(
    document_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)
ques = 'What are the applications of attention mechanism?'

document_chain_with_hist.invoke(
    {"input": ques},
    config={ "configurable": {"session_id": session_id}},  # constructs a key "abc123" in `store`.
)



KeyError: 'context'

In [70]:
from langchain_community.document_loaders.notebook import remove_newlines
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('../playground/ip_data/attention-is-all-you-need.pdf')
documents = loader.load()

splits = text_splitter.split_documents(documents)

cleaned_docs = [
    Document(page_content=remove_newlines(doc.page_content), metadata=doc.metadata)
    for doc in splits
]


prompt = ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>


Chat History:
{chat_history}

Question: {input}

"""
)

document_chain=create_stuff_documents_chain(llm,prompt)
# ques = 'How many people are there in the story ?'
# ques = 'Should Clara follow the young man or the older man ?'
ques = 'Summarize the content'

response = document_chain.invoke({
"input" : ques,
"context" :cleaned_docs
})



# Trying to create history for the above pipeline  :
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

session_id = 'abc123'

document_chain_with_hist = RunnableWithMessageHistory(
    document_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)
ques = 'What are the applications of attention mechanism?'

document_chain_with_hist.invoke(
    {"input": ques},
    config={ "configurable": {"session_id": session_id}},  # constructs a key "abc123" in `store`.
)



{'abc123': InMemoryChatMessageHistory(messages=[])}

In [77]:
from langchain_community.document_loaders.notebook import remove_newlines
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains.combine_documents import create_stuff_documents_chain

# Load and process documents
loader = PyPDFLoader('../playground/ip_data/attention-is-all-you-need.pdf')
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(documents)

cleaned_docs = [
    Document(page_content=remove_newlines(doc.page_content), metadata=doc.metadata)
    for doc in splits
]

# Updated prompt with chat_history placeholder
prompt = ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:

<context>
{context}
</context>

Chat History:
{chat_history}

Question: {input}
"""
)

# Create the base document chain
document_chain = create_stuff_documents_chain(llm, prompt)

# Message history storage and retrieval
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Wrap with message history
document_chain_with_hist = RunnableWithMessageHistory(
    document_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

# Test the chain
session_id = 'abc123'

# First question
response1 = document_chain_with_hist.invoke(
    {
        "input": "What are the applications of attention mechanism?",
        "context": cleaned_docs
    },
    config={"configurable": {"session_id": session_id}},
)
print("Response 1:", response1)

# Follow-up question (will have access to chat history)
response2 = document_chain_with_hist.invoke(
    {
        "input": "Can you elaborate more on that?",
        "context": cleaned_docs
    },
    config={"configurable": {"session_id": session_id}},
)
print("Response 2:", response2)

# Check stored history
print("Chat History:", store[session_id].messages)

Response 1: The passage does not provide any information about the applications of the attention mechanism, so I cannot answer this question from the provided context.
Response 2: The passage does not provide any information about the applications of the attention mechanism, so I cannot elaborate more on that.
Chat History: [HumanMessage(content='What are the applications of attention mechanism?', additional_kwargs={}, response_metadata={}), AIMessage(content='The passage does not provide any information about the applications of the attention mechanism, so I cannot answer this question from the provided context.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Can you elaborate more on that?', additional_kwargs={}, response_metadata={}), AIMessage(content='The passage does not provide any information about the applications of the attention mechanism, so I cannot elaborate more on that.', additional_kwargs={}, response_metadata={})]


In [78]:
cleaned_docs

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 